# E-Governance Service Analytics
## Yuva Intern — Week 1 | Synthetic demonstration

**Important:** This notebook uses generated sample records, not real government data. Findings are illustrative only.

### Questions
- What is the distribution of application statuses?
- Which services have the largest backlog?
- How long do completed applications take in this synthetic sample?
- What rejection and complaint categories occur most often?

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Works when run from the repository root or from the notebooks/ folder.
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "e_governance_applications_synthetic.csv"

df = pd.read_csv(DATA_PATH)
df["submission_date"] = pd.to_datetime(df["submission_date"], errors="coerce")
df["completion_date"] = pd.to_datetime(df["completion_date"], errors="coerce")
df["processing_days"] = pd.to_numeric(df["processing_days"], errors="coerce")
df["satisfaction_rating"] = pd.to_numeric(df["satisfaction_rating"], errors="coerce")

print("Shape:", df.shape)
df.head()

## 1. Data quality checks

In [ ]:
quality = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str)
}).sort_values("missing_pct", ascending=False)
display(quality)
print("Duplicate application IDs:", df["application_id"].duplicated().sum())
print("Unique application IDs:", df["application_id"].nunique())
print("Synthetic flag values:", df["is_synthetic"].unique())

## 2. Status distribution

In [ ]:
status_counts = df["status"].value_counts().rename_axis("status").reset_index(name="applications")
display(status_counts)
ax = status_counts.plot(kind="bar", x="status", y="applications", legend=False, figsize=(7, 4))
ax.set_title("Application Status Distribution — Synthetic Demo")
ax.set_xlabel("")
ax.set_ylabel("Applications")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Key performance indicators

In [ ]:
total = df["application_id"].nunique()
completed = (df["status"] == "Completed").sum()
rejected = (df["status"] == "Rejected").sum()
decided = df["status"].isin(["Completed", "Rejected"]).sum()
backlog = df["status"].isin(["Pending", "In Review"]).sum()
completed_days = df.loc[df["status"] == "Completed", "processing_days"]

kpis = pd.DataFrame([{
    "Total applications": total,
    "Completion rate (%)": round(completed / total * 100, 2) if total else np.nan,
    "Rejection rate among decided (%)": round(rejected / decided * 100, 2) if decided else np.nan,
    "Pending + in-review backlog": int(backlog),
    "Median processing days": round(completed_days.median(), 2),
    "90th percentile processing days": round(completed_days.quantile(.90), 2),
    "Average satisfaction (available ratings)": round(df["satisfaction_rating"].mean(), 2)
}])
display(kpis.T.rename(columns={0: "Value"}))

## 4. Compare service types

In [ ]:
service_summary = df.groupby("service_type").agg(
    applications=("application_id", "nunique"),
    completed=("status", lambda x: (x == "Completed").sum()),
    rejected=("status", lambda x: (x == "Rejected").sum()),
    backlog=("status", lambda x: x.isin(["Pending", "In Review"]).sum()),
    median_processing_days=("processing_days", "median")
).sort_values("applications", ascending=False)
display(service_summary)
service_summary["applications"].sort_values().plot(kind="barh", figsize=(8, 5), title="Applications by Service Type")
plt.xlabel("Applications")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 5. Rejection and complaint themes

In [ ]:
rejection_summary = (
    df.loc[df["status"] == "Rejected", "rejection_reason"]
      .value_counts().rename_axis("rejection_reason").reset_index(name="count")
)
complaint_summary = (
    df.loc[df["complaint_category"] != "None", "complaint_category"]
      .value_counts().rename_axis("complaint_category").reset_index(name="count")
)
display(rejection_summary)
display(complaint_summary)

## 6. Monthly application trend

In [ ]:
monthly = df.assign(month=df["submission_date"].dt.to_period("M").astype(str)).groupby("month").size()
monthly.plot(figsize=(10, 4), marker="o", title="Monthly Applications — Synthetic Demo")
plt.xlabel("Submission month")
plt.ylabel("Applications")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

## 7. Findings and limitations

Write 3–5 findings here **after running the notebook**, using the actual values displayed above. Do not copy assumed numbers.

### Interpretation checklist
- These patterns describe only generated sample data.
- A group difference does not prove the channel or district caused the outcome.
- Satisfaction ratings exist only for a subset of records; response bias is possible.
- In real use, validate KPI definitions, service timelines, data coverage, and source quality with authorised service owners.
- Protect personal data and publish aggregated results only.

In [ ]:
# Optional: export summary tables for reporting or Power BI
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
kpis.to_csv(OUT / "kpi_summary.csv", index=False)
service_summary.to_csv(OUT / "service_summary.csv")
rejection_summary.to_csv(OUT / "rejection_reasons.csv", index=False)
complaint_summary.to_csv(OUT / "complaint_categories.csv", index=False)
print("Saved summary files to", OUT)